# VisionAssist Phase 11 — Hard-example adapter iteration

This notebook rebuilds the validation-driven 6,000-record selection, initializes from the promoted Phase 10 adapter with a fresh optimizer, runs a GPU smoke gate, trains resumably, and evaluates against unchanged validation and frozen-test records. Run cells in order in a fresh A100 runtime.

In [ ]:
#@title 1. Settings — run before importing Torch
import os
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/moshiur00/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")
DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
PILOT_RUN_ID = "qwen25vl3b_qlora_pilot_v1"
HARD_RUN_ID = "qwen25vl3b_qlora_hard_examples_v1"
HARD_SELECTION_CONFIG = PROJECT_ROOT / "configs/training/phase11_hard_examples.yaml"
HARD_TRAINING_CONFIG = PROJECT_ROOT / "configs/training/qwen25vl3b_qlora_hard_examples.yaml"

In [ ]:
#@title 2. Mount Drive and verify required artifacts
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PILOT = DRIVE_ROOT / "outputs/training" / PILOT_RUN_ID
DRIVE_PILOT_EVAL = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_pilot_best"
required_drive = [DRIVE_DATA_ARCHIVE, DRIVE_PILOT / "final_adapter/adapter_model.safetensors", DRIVE_PILOT_EVAL / "validation/predictions.jsonl"]
for path in required_drive: print(path, path.exists())
assert all(path.exists() for path in required_drive)

In [ ]:
#@title 3. Clone/update repository and install dependencies
import shutil, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["uv", "sync", "--extra", "training", "--extra", "dev"], cwd=PROJECT_ROOT, check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

In [ ]:
#@title 4. Verify GPU
import torch
assert torch.cuda.is_available(), "Select a GPU runtime."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(properties.total_memory / 1024**3, 2))
print("BF16:", torch.cuda.is_bf16_supported())
subprocess.run(["nvidia-smi"], check=False)

## A. Restore data and the promoted Phase 10 evidence

In [ ]:
#@title 5. Restore prepared data and Phase 10 artifacts
import tarfile
required_data = [PROJECT_ROOT / "data/raw/visa", PROJECT_ROOT / "data/processed/visa_instructions/train.jsonl", PROJECT_ROOT / "data/processed/visa_instructions/validation.jsonl", PROJECT_ROOT / "data/processed/visa_instructions/test.jsonl", PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"]
if not all(path.exists() for path in required_data):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive: archive.extractall(PROJECT_ROOT, filter="data")
assert all(path.exists() for path in required_data)
local_pilot = PROJECT_ROOT / "outputs/training" / PILOT_RUN_ID
local_pilot_eval = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_pilot_best"
shutil.copytree(DRIVE_PILOT, local_pilot, dirs_exist_ok=True)
shutil.copytree(DRIVE_PILOT_EVAL, local_pilot_eval, dirs_exist_ok=True)
print("Data and promoted adapter restored.")

In [ ]:
#@title 6. Normalize instruction and benchmark image paths
import hashlib, json
from pathlib import PurePosixPath
MARKER = ("data", "raw", "visa")
def normalize_image_path(value):
    parts = PurePosixPath(str(value).replace("\\", "/")).parts
    lowered = tuple(part.lower() for part in parts)
    for index in range(len(parts) - len(MARKER) + 1):
        if lowered[index:index + len(MARKER)] == MARKER: return PurePosixPath(*parts[index:]).as_posix()
    candidate = PurePosixPath(*parts)
    if not candidate.is_absolute() and ".." not in candidate.parts: return candidate.as_posix()
    raise ValueError(value)
def normalize_jsonl(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        row = json.loads(line)
        for message in row.get("messages", []):
            if message.get("role") != "user": continue
            for item in message.get("content", []):
                if item.get("type") == "image": item["image"] = normalize_image_path(item["image"])
        rows.append(row)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
    temporary.replace(path)
instruction_root = PROJECT_ROOT / "data/processed/visa_instructions"
for split in ("train", "validation", "test"): normalize_jsonl(instruction_root / f"{split}.jsonl")
benchmark = PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"
normalize_jsonl(benchmark)
benchmark_hash = hashlib.sha256(benchmark.read_bytes()).hexdigest()
manifest_path = benchmark.parent / "benchmark_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")); manifest["benchmark_sha256"] = benchmark_hash
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(benchmark.parent / "benchmark_sha256.txt").write_text(benchmark_hash + "\n", encoding="utf-8")

In [ ]:
#@title 7. Run Phase 11 CPU regression tests
subprocess.run(["uv", "run", "pytest", "tests/test_phase8_training.py", "tests/test_phase11_failure_analysis.py", "tests/test_phase11_hard_examples.py"], cwd=PROJECT_ROOT, check=True)

## B. Rebuild and audit the leakage-safe selection

In [ ]:
#@title 8. Reproduce the 6,000-record hard-example selection
import yaml
subprocess.run(["uv", "run", "visionassist", "select-hard-examples", "--config", str(HARD_SELECTION_CONFIG)], cwd=PROJECT_ROOT, check=True)
selection_config = yaml.safe_load(HARD_SELECTION_CONFIG.read_text(encoding="utf-8"))
selection_manifest_path = PROJECT_ROOT / selection_config["manifest_path"]
selection_manifest = json.loads(selection_manifest_path.read_text(encoding="utf-8"))
print(json.dumps(selection_manifest, indent=2))
assert selection_manifest["records"] == 6000
assert selection_manifest["unique_instruction_ids"] == 6000
assert selection_manifest["task_counts"] == selection_manifest["task_quotas"]
assert min(value for counts in selection_manifest["task_category_counts"].values() for value in counts.values()) >= 20
assert selection_manifest["leakage"] == {"validation_image_overlap": 0, "test_image_overlap": 0}
assert selection_manifest["instruction_ids_sha256"] == "440390b2a4ab6b5491eeaba806f5c5b45a9f460781bd67d9008fe13d33e1e3e6"

In [ ]:
#@title 9. Configure Drive checkpoints and verify adapter initialization
training = yaml.safe_load(HARD_TRAINING_CONFIG.read_text(encoding="utf-8"))
training["checkpoints"]["persistent_output_dir"] = str(DRIVE_ROOT / "checkpoints" / HARD_RUN_ID)
training["checkpoints"]["sync_every_save"] = True
HARD_TRAINING_CONFIG.write_text(yaml.safe_dump(training, sort_keys=False), encoding="utf-8")
adapter = PROJECT_ROOT / training["initial_adapter_path"]
assert (adapter / "adapter_config.json").is_file()
assert (adapter / "adapter_model.safetensors").is_file()
checkpoints = DRIVE_ROOT / "checkpoints" / HARD_RUN_ID
print("Existing Phase 11 checkpoints:", [p.name for p in sorted(checkpoints.glob("checkpoint-*"))] if checkpoints.is_dir() else [])
subprocess.run(["uv", "run", "visionassist", "training-environment", "--config", str(HARD_TRAINING_CONFIG)], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 10. One-batch promoted-adapter smoke test
subprocess.run(["uv", "run", "visionassist", "training-smoke-test", "--config", str(HARD_TRAINING_CONFIG)], cwd=PROJECT_ROOT, check=True)
smoke_path = PROJECT_ROOT / "outputs/training" / HARD_RUN_ID / "one_batch_smoke_test.json"
smoke = json.loads(smoke_path.read_text(encoding="utf-8"))
print(json.dumps(smoke, indent=2))
assert smoke["passed"] and smoke["finite_gradients"] and smoke["nonzero_gradients"]

## C. Explicit Phase 11 training gate

Confirm the selection fingerprint, zero leakage, adapter files, checkpoint ownership, and smoke report before opening the gate. This run starts a fresh optimizer from the promoted adapter; `--resume latest` later resumes only Phase 11 checkpoints.

In [ ]:
#@title 11. Start or resume Phase 11 training
START_HARD_EXAMPLE_TRAINING = False
if not START_HARD_EXAMPLE_TRAINING: raise RuntimeError("Training gate is closed. Review Cells 8–10 first.")
subprocess.run(["uv", "run", "visionassist", "train-qlora", "--config", str(HARD_TRAINING_CONFIG), "--resume", "latest"], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 12. Verify and persist completed Phase 11 adapter
hard_run = PROJECT_ROOT / "outputs/training" / HARD_RUN_ID
run_manifest = json.loads((hard_run / "run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(run_manifest, indent=2))
assert run_manifest["status"] == "completed"
assert run_manifest["initial_adapter_path"] == training["initial_adapter_path"]
assert (hard_run / "final_adapter/adapter_model.safetensors").is_file()
drive_hard_run = DRIVE_ROOT / "outputs/training" / HARD_RUN_ID
shutil.copytree(hard_run, drive_hard_run, dirs_exist_ok=True)
print("Saved Phase 11 artifacts to:", drive_hard_run)

## D. Gated validation and frozen-test evaluation

In [ ]:
#@title 13. Create Drive-resumable evaluation configs
base_dir = PROJECT_ROOT / "configs/inference"
evaluation_configs = {}
for split_name, source_name, limit, seed in (("validation", "qwen25vl3b_overfit_checkpoint50_validation.yaml", 1000, 43), ("test", "qwen25vl3b_overfit_checkpoint50_test.yaml", None, 44)):
    config = yaml.safe_load((base_dir / source_name).read_text(encoding="utf-8"))
    output = f"outputs/post_training/qwen25vl3b_hard_examples_best/{split_name}"
    config.update({"run_id": f"qwen25vl3b_hard_examples_best_{split_name}_v1", "adapter_path": f"outputs/training/{HARD_RUN_ID}/final_adapter", "output_dir": output, "partial_predictions_path": f"{output}/predictions.partial.jsonl", "predictions_path": f"{output}/predictions.jsonl", "errors_path": f"{output}/inference_errors.jsonl", "run_manifest_path": f"{output}/run_manifest.json", "evaluation_records_path": f"{output}/evaluation_records.jsonl", "subset_limit": limit, "subset_seed": seed, "overwrite": False, "persistent_output_dir": str(DRIVE_ROOT / "inference" / f"qwen25vl3b_hard_examples_best_{split_name}"), "persistent_sync_every": 25})
    if split_name == "validation": config["benchmark_manifest_path"] = f"outputs/training/{HARD_RUN_ID}/dataset_manifest.json"
    path = base_dir / f"qwen25vl3b_hard_examples_best_{split_name}.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    evaluation_configs[split_name] = path
    print(split_name, path)

In [ ]:
#@title 14. Run validation first
RUN_VALIDATION = False
if not RUN_VALIDATION: raise RuntimeError("Validation gate is closed.")
subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", str(evaluation_configs["validation"])], cwd=PROJECT_ROOT, check=True)
torch.cuda.empty_cache()

In [ ]:
#@title 15. Review validation against the promoted Phase 10 adapter
new_validation = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_hard_examples_best/validation/evaluation/metrics.json"
old_validation = local_pilot_eval / "validation/evaluation/metrics.json"
for label, path in (("PHASE 10", old_validation), ("PHASE 11", new_validation)):
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print(f"\n===== {label} VALIDATION =====")
    print("failure_rate", metrics["failure_rate"])
    for task, values in metrics["per_task"].items(): print(task, {k: v for k, v in values.items() if k not in {"per_label", "confusion_matrix"}})

In [ ]:
#@title 16. Run the complete frozen test only after validation review
RUN_FROZEN_TEST = False
if not RUN_FROZEN_TEST: raise RuntimeError("Frozen-test gate is closed. Review validation first.")
subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", str(evaluation_configs["test"])], cwd=PROJECT_ROOT, check=True)
torch.cuda.empty_cache()

In [ ]:
#@title 17. Summarize and persist Phase 11 evaluations
assessment = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_hard_examples_best"
for split_name in ("validation", "test"):
    run_dir = assessment / split_name
    if not (run_dir / "assessment_summary.json").is_file(): continue
    summary = json.loads((run_dir / "assessment_summary.json").read_text(encoding="utf-8"))
    metrics = json.loads((run_dir / "evaluation/metrics.json").read_text(encoding="utf-8"))
    print(f"\n===== {split_name.upper()} ====="); print(json.dumps(summary, indent=2)); print("failure tags:", metrics.get("failure_tag_counts", {}))
drive_assessment = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_hard_examples_best"
shutil.copytree(assessment, drive_assessment, dirs_exist_ok=True)
print("Saved evaluation artifacts to:", drive_assessment)

## Resume and monitoring

After a disconnect, rerun setup, restore data/artifacts, regenerate the selection, configure persistence, and reopen only the needed gate. Training resumes from the newest checkpoint belonging to `qwen25vl3b_qlora_hard_examples_v1`. Inference restores Drive-synchronized partial predictions every 25 records. Do not run the frozen test until validation has been compared with Phase 10.